# Projeto AURORA — Sistema de Verificação de Decolagem

**Atividade integradora (PBL) — FIAP**

Este notebook reúne os itens 1.1 a 1.6 da atividade. Ele é executável de ponta a
ponta: basta rodar as células na ordem (menu *Run > Run All Cells*).

| Item | Conteúdo |
| :--- | :--- |
| 1.1 | Organização e descrição da telemetria |
| 1.2 | Algoritmo de verificação |
| 1.3 | Script em Python |
| 1.4 | Análise energética |
| 1.5 | Análise assistida por IA |
| 1.6 | Reflexão crítica |

> **Sobre a estrutura do projeto:** a versão interativa do programa, que lê os
> dados digitados pelo operador, está em `scripts/main.py`. Neste notebook os
> dados são definidos em variáveis para que a execução seja reprodutível e não
> dependa de digitação.

---
## 1.1 Organização e descrição da telemetria

O sistema monitora seis parâmetros. A distinção entre eles importa para o desenho
do algoritmo: variáveis **contínuas** admitem faixa de tolerância; variáveis
**binárias** não admitem meio-termo — ou estão no valor esperado, ou são aborto
imediato.

| Parâmetro | Natureza | Unidade | Faixa segura |
| :--- | :--- | :--- | :--- |
| Temperatura interna | Quantitativa contínua | °C | 15 a 30 |
| Temperatura externa | Quantitativa contínua | °C | −10 a 45 |
| Integridade estrutural | Qualitativa binária | — | apenas 1 |
| Pressão dos tanques | Quantitativa contínua | psi | 450 a 550 |
| Nível de energia | Quantitativa contínua | % | mínimo de 80 |
| Módulos críticos | Qualitativa binária | — | todos online |

### Classificação por criticidade

| Classe | Parâmetros | Característica da falha |
| :--- | :--- | :--- |
| **A — Catastrófica** | Integridade, Módulos críticos | Binária, súbita, sem aviso prévio |
| **B — Progressiva** | Pressão, Energia | Degrada ao longo do voo |
| **C — Ambiental** | Temperaturas | Degrada componentes lentamente |

Essa classificação justifica a ordem das verificações: nenhuma leitura posterior
tem valor se a estrutura já está comprometida.

---
## 1.2 Algoritmo de verificação

O algoritmo trabalha em três fases:

**FASE 1 — Verificações de segurança.** As seis verificações são executadas
**sempre**, e cada uma pode desligar a chave `decolagem_autorizada`. Elas não
abortam em cascata: o operador precisa enxergar todos os problemas de uma vez, e
não descobrir o próximo defeito só na tentativa seguinte.

**FASE 2 — Análise assistida por IA.** Cruza os parâmetros entre si, procurando
discrepâncias que nenhuma verificação isolada consegue detectar.

**FASE 3 — Veredito e classificação.**

| Chave | Alertas | Classificação |
| :--- | :--- | :--- |
| Desligada | — | **HORRÍVEL** — decolagem abortada |
| Ligada | 1 ou mais | **MÉDIO** — autorizada com ressalvas |
| Ligada | nenhum | **ÓTIMO** — sucesso absoluto |

> Os fluxogramas completos, em formato Mermaid, estão no arquivo `ROADMAP.MD` do
> repositório, onde o GitHub os renderiza graficamente.

---
## 1.3 Script em Python

### Leitura dos dados

No programa interativo (`scripts/main.py`) esta etapa usa `input()`. Aqui os
valores são fixos para tornar a execução reprodutível.

Altere os valores da célula abaixo para simular outros cenários.

In [ ]:
# --- TELEMETRIA RECEBIDA ---
# No programa interativo estes valores vem de input(); aqui sao fixos.
temp_interna = 29.0      # graus Celsius
temp_externa = -8.0      # graus Celsius
integridade = 1          # 1 = OK, 0 = falha
pressao_tanques = 535.0  # psi
energia = 93.0           # porcentagem da bateria
modulos_online = True    # todos os modulos criticos online

capacidade_total = 1000.0  # kWh - capacidade nominal do banco de baterias

# A CHAVE COMECA LIGADA E SO PODE SER DESLIGADA PELAS VERIFICACOES
decolagem_autorizada = True

print("Telemetria carregada.")

### Execução das verificações

Cada bloco compara um parâmetro com sua faixa segura. Note que são `if`
independentes, e não `elif`: todos são avaliados.

In [ ]:
# Verificacao 1 - TEMPERATURAS
if temp_interna > 30 or temp_interna < 15:
    print("Erro: Temperatura Interna fora do range!")
    decolagem_autorizada = False
else:
    print("Temperatura Interna: OK")

if temp_externa > 45 or temp_externa < -10:
    print("Erro: Temperatura Externa fora do range!")
    decolagem_autorizada = False
else:
    print("Temperatura Externa: OK")

# Verificacao 2 - INTEGRIDADE
if integridade < 1:
    print("Erro: Integridade comprometida!")
    decolagem_autorizada = False
else:
    print("Integridade: OK")

# Verificacao 3 - PRESSAO DOS TANQUES
if pressao_tanques < 450 or pressao_tanques > 550:
    print("Erro: Pressao dos tanques fora do range!")
    decolagem_autorizada = False
else:
    print("Pressao dos Tanques: OK")

---
## 1.4 Análise energética

O nível de energia é informado em **porcentagem**, mas a autonomia precisa ser
calculada em **kWh**. O cálculo tem três passos:

1. Converter a carga atual de % para kWh
2. Somar as perdas ao consumo estimado da decolagem
3. Subtrair um do outro para obter a reserva

As perdas são aplicadas **sobre o consumo** (conversão do inversor e aquecimento),
não sobre a bateria.

In [ ]:
# CONSTANTES DO PROJETO
CONSUMO_DECOLAGEM = 300.0   # kWh estimados para a fase de decolagem
PERDAS = 0.08               # 8% de perdas por conversao e aquecimento
RESERVA_MINIMA = 0.10       # 10% da capacidade deve sobrar para manobra e pouso

energia_disponivel = capacidade_total * (energia / 100)
consumo_real = CONSUMO_DECOLAGEM * (1 + PERDAS)
energia_restante = energia_disponivel - consumo_real
autonomia_restante = (energia_restante / capacidade_total) * 100

print("Disponivel na bateria : {:.1f} kWh ({:.0f}%)".format(energia_disponivel, energia))
print("Consumo + perdas      : {:.1f} kWh".format(consumo_real))
print("Sobra apos decolagem  : {:.1f} kWh ({:.1f}%)".format(energia_restante, autonomia_restante))

### Verificação de energia e de reserva

A verificação de reserva só faz sentido se a energia mínima já tiver passado — por
isso é um `elif`, e não um `if` independente.

Um detalhe que vale registrar: com consumo de 300 kWh e 8% de perdas, a energia
mínima *matemática* para decolar seria de apenas **32,4%**. O limite de 80% não vem
do consumo da decolagem — ele existe para garantir autonomia **depois** dela.

In [ ]:
# Verificacao 4 - ENERGIA
if energia < 80:
    print("Erro: Energia insuficiente!")
    decolagem_autorizada = False
elif energia_restante < capacidade_total * RESERVA_MINIMA:
    print("Erro: Reserva pos-decolagem insuficiente ({:.1f} kWh)!".format(energia_restante))
    decolagem_autorizada = False
else:
    print("Energia: OK")

# Verificacao 5 - MODULOS ONLINE
if not modulos_online:
    print("Erro: Modulos offline!")
    decolagem_autorizada = False
else:
    print("Modulos: OK")

---
## 1.5 Análise assistida por IA

As verificações acima olham cada parâmetro **sozinho**. Esta etapa cruza os dados
entre si, procurando discrepâncias que nenhuma verificação isolada consegue
enxergar.

É um **sistema especialista**: a inteligência está nas regras de correlação entre
os sensores. A opção por regras locais, em vez de uma chamada a um modelo de
linguagem externo, foi deliberada — o sistema precisa funcionar sem internet, sem
chave de API e de forma determinística, produzindo sempre o mesmo veredito para a
mesma entrada. Em um sistema que decide sobre segurança, reprodutibilidade é
requisito.

As regras estão organizadas em quatro grupos:

| Grupo | Pergunta que responde |
| :--- | :--- |
| 1 | Os dados são fisicamente possíveis? |
| 2 | Os dados são coerentes entre si? |
| 3 | Algum parâmetro opera sem margem de segurança? |
| 4 | A telemetria parece real? |

In [ ]:
criticos = []   # DADOS IMPOSSIVEIS (TELEMETRIA NAO CONFIAVEL)
alertas = []    # COMBINACOES DE RISCO (DADOS VALIDOS, MAS PERIGOSOS)

# --- GRUPO 1: OS DADOS SAO FISICAMENTE POSSIVEIS? ---
if energia > 100 or energia < 0:
    criticos.append("Energia de {:.1f}% esta fora do dominio fisico (0 a 100%).".format(energia))

if integridade != 0 and integridade != 1:
    criticos.append("Integridade informada como {}. O indicador e binario (0 ou 1).".format(integridade))

if capacidade_total <= 0:
    criticos.append("Capacidade da bateria informada como {:.1f} kWh. Valor impossivel.".format(capacidade_total))

if pressao_tanques < 0:
    criticos.append("Pressao negativa ({:.1f} psi) e fisicamente impossivel.".format(pressao_tanques))

print("Criticos encontrados:", len(criticos))

In [ ]:
# --- GRUPO 2: OS DADOS SAO COERENTES ENTRE SI? ---
diferenca_termica = abs(temp_interna - temp_externa)

if temp_externa < 0 and energia < 90:
    alertas.append("Energia em {:.0f}% com temperatura externa de {:.0f}C. Baterias de litio "
                   "perdem capacidade UTILIZAVEL no frio.".format(energia, temp_externa))

if pressao_tanques > 520 and temp_interna > 27:
    alertas.append("Pressao de {:.0f} psi ja alta com temperatura interna de {:.0f}C. Pela lei dos "
                   "gases a pressao sobe com o aquecimento.".format(pressao_tanques, temp_interna))

if diferenca_termica > 35:
    alertas.append("Diferencial termico de {:.0f}C entre interna e externa. Sugere falha de "
                   "isolamento ou sensor travado.".format(diferenca_termica))

# --- GRUPO 3: ALGUM PARAMETRO OPERA SEM MARGEM? ---
if 80 <= energia <= 82:
    alertas.append("Energia a {:.1f}%, no limiar dos 80% exigidos.".format(energia))

if 450 <= pressao_tanques <= 460 or 540 <= pressao_tanques <= 550:
    alertas.append("Pressao de {:.0f} psi opera na fronteira da faixa segura.".format(pressao_tanques))

if temp_interna >= 28:
    alertas.append("Temperatura interna de {:.0f}C proxima do teto de 30C.".format(temp_interna))

if 0 < autonomia_restante < 20:
    alertas.append("Reserva pos-decolagem de apenas {:.1f}% da bateria.".format(autonomia_restante))

# --- GRUPO 4: A TELEMETRIA PARECE REAL? ---
if temp_interna == 22 and temp_externa == 25 and pressao_tanques == 500 and energia == 100:
    alertas.append("Todos os canais retornaram exatamente o valor nominal. Em hardware real isso "
                   "e estatisticamente improvavel.")

print("Alertas encontrados:", len(alertas))

### Relatório da análise e veredito final

In [ ]:
print("=" * 62)
print("ANALISE ASSISTIDA POR IA - DIAGNOSTICO DE DISCREPANCIAS")
print("=" * 62)

if criticos:
    print("\nDISCREPANCIAS CRITICAS (telemetria nao confiavel):")
    for item in criticos:
        print("  [X] " + item)

if alertas:
    print("\nALERTAS (dados validos, mas em combinacao de risco):")
    for item in alertas:
        print("  [!] " + item)

print("")
if criticos:
    print(">> PARECER: DADOS INCONSISTENTES.")
    decolagem_autorizada = False
    classificacao = "HORRIVEL"
elif not decolagem_autorizada:
    print(">> PARECER: DECOLAGEM JA BLOQUEADA pelas verificacoes de seguranca.")
    classificacao = "HORRIVEL"
elif alertas:
    print(">> PARECER: DECOLAGEM VIAVEL, COM RESSALVAS.")
    print("   {} ponto(s) de atencao. Recomenda-se revisao humana.".format(len(alertas)))
    classificacao = "MEDIO"
else:
    print(">> PARECER: SUCESSO ABSOLUTO.")
    classificacao = "OTIMO"

print("=" * 62)
if decolagem_autorizada:
    print("VEREDITO FINAL: Decolagem Autorizada!  (classificacao: {})".format(classificacao))
else:
    print("VEREDITO FINAL: Decolagem Nao Autorizada!  (classificacao: {})".format(classificacao))

---
## Coleta e análise dos cenários

O programa interativo grava cada execução na pasta `cenarios/`: uma planilha CSV
acumulativa e um relatório individual em TXT. Foram coletados **10 cenários**,
gerados pelo script `scripts/cenarios.py`.

A célula abaixo lê a planilha e monta a tabela comparativa.

In [ ]:
import csv
import os

# PROCURA A PLANILHA TANTO A PARTIR DA PASTA DO NOTEBOOK QUANTO DA RAIZ DO PROJETO
caminhos_possiveis = [
    os.path.join("..", "cenarios", "registro_execucoes.csv"),
    os.path.join("cenarios", "registro_execucoes.csv"),
]

arquivo_csv = None
for caminho in caminhos_possiveis:
    if os.path.exists(caminho):
        arquivo_csv = caminho
        break

if arquivo_csv is None:
    print("Planilha nao encontrada. Rode antes: python scripts/cenarios.py")
    registros = []
else:
    arquivo = open(arquivo_csv, "r", encoding="utf-8-sig")
    registros = list(csv.DictReader(arquivo, delimiter=";"))
    arquivo.close()
    print("Cenarios carregados de {}: {}".format(arquivo_csv, len(registros)))

In [ ]:
# TABELA COMPARATIVA DOS CENARIOS
# CRIT = discrepancias criticas (bloqueiam) | ALRT = alertas (nao bloqueiam sozinhos)
if registros:
    print("{:>3}  {:<9}  {:>6}  {:>6}  {:>4}  {:>6}  {:>6}  {:>4}  {:>4}  {:>7}".format(
        "#", "CLASSE", "T.INT", "T.EXT", "INTG", "PRESS", "ENERG", "CRIT", "ALRT", "RESERVA"))
    print("-" * 76)
    for r in registros:
        print("{:>3}  {:<9}  {:>6}  {:>6}  {:>4}  {:>6}  {:>6}  {:>4}  {:>4}  {:>6}%".format(
            r["execucao"], r["classificacao"], r["temp_interna"], r["temp_externa"],
            r["integridade"], r["pressao_psi"], r["energia_pct"],
            r["qtd_criticos"], r["qtd_alertas"], r["autonomia_pct"]))

In [ ]:
# DISTRIBUICAO DAS CLASSIFICACOES
if registros:
    contagem = {"OTIMO": 0, "MEDIO": 0, "HORRIVEL": 0}
    for r in registros:
        contagem[r["classificacao"]] = contagem.get(r["classificacao"], 0) + 1

    print("DISTRIBUICAO DOS {} CENARIOS COLETADOS\n".format(len(registros)))
    for classe in ["OTIMO", "MEDIO", "HORRIVEL"]:
        quantidade = contagem[classe]
        barra = "#" * (quantidade * 4)
        print("{:<9} {:>2}  {}".format(classe, quantidade, barra))

In [ ]:
# GRAFICO (OPCIONAL): SO EXECUTA SE matplotlib ESTIVER INSTALADO.
# SEM A BIBLIOTECA, AS CELULAS ANTERIORES JA MOSTRARAM OS MESMOS DADOS EM TEXTO.
try:
    import matplotlib.pyplot as plt

    if registros:
        cores = {"OTIMO": "#2e7d32", "MEDIO": "#f9a825", "HORRIVEL": "#c62828"}
        execucoes = [int(r["execucao"]) for r in registros]
        energias = [float(r["energia_pct"]) for r in registros]
        cor_barras = [cores.get(r["classificacao"], "#666666") for r in registros]

        figura, eixo = plt.subplots(figsize=(10, 4))
        eixo.bar(execucoes, energias, color=cor_barras)
        eixo.axhline(80, color="black", linestyle="--", linewidth=1, label="Minimo de 80%")
        eixo.set_xlabel("Cenario")
        eixo.set_ylabel("Energia informada (%)")
        eixo.set_title("Energia por cenario (cor = classificacao)")
        eixo.set_xticks(execucoes)
        eixo.legend()
        plt.tight_layout()
        plt.show()
except ImportError:
    print("matplotlib nao instalado - grafico omitido.")
    print("Para instalar: pip install matplotlib")

### Leitura dos resultados

O cenário mais instrutivo da coleta é o **número 08**, com energia informada como
105% e integridade como 3.

As seis verificações clássicas aprovaram todos os parâmetros — afinal, 105 não é
menor que 80, e 3 não é menor que 1. O sistema teria autorizado a decolagem de uma
nave com sensores corrompidos. Foi a camada de análise que barrou.

Repare na coluna de energia disponível desse cenário na planilha: **1050 kWh em uma
bateria de 1000 kWh**. É a prova numérica de que o dado era impossível — algo que
nenhuma verificação de faixa isolada conseguiria perceber, porque cada valor,
sozinho, estava dentro do range.

O cenário **09** cumpre o papel oposto e igualmente importante: com temperatura
externa de 5°C, ele **não** dispara o alerta de frio. Isso demonstra que a regra
discrimina de fato, em vez de alertar sobre qualquer coisa.

---
## 1.6 Reflexão crítica

O texto completo está em `reflexao_critica.md`, no repositório. Os pontos centrais:

**Ética e responsabilidade.** Não é o algoritmo que autoriza a decolagem — são as
pessoas que definiram os limites. Os valores usados (80% de energia, 450–550 psi,
8% de perdas) foram escolhidos por nós e não validados em bancada. A automação não
transfere responsabilidade: concentra-a no momento do projeto. O cenário 08 mostrou
que um sistema de verificação pode transmitir mais segurança do que efetivamente
oferece.

**Impacto social.** A capacidade de lançamento está concentrada em poucos países e
empresas, enquanto a órbita é um bem comum e finito. Vale lembrar que o critério de
abortar protege também as comunidades sob a trajetória — pessoas que não participam
da decisão e raramente se beneficiam dela.

**Sustentabilidade tecnológica.** Propulsão elétrica não é automaticamente limpa: a
origem da eletricidade e a mineração de lítio fazem parte da conta. Reduzir os 8%
de perdas é ganho ambiental e de autonomia ao mesmo tempo. E há a sustentabilidade
do próprio software: em sistemas que decidem sobre vidas, débito técnico não custa
tempo — custa segurança.

---
## Como executar

**Programa interativo** (pede os dados pelo teclado e grava o cenário):

```
python scripts/main.py
```

**Gerador de cenários** (executa os cenários pré-definidos que faltam):

```
python scripts/cenarios.py
```

**Este notebook**: execute as células em ordem.

## Estrutura do repositório

```
├── scripts/
│   ├── main.py          programa principal (interativo)
│   └── cenarios.py      gerador automatico de cenarios
├── cenarios/            10 cenarios coletados (CSV + TXT)
├── notebook/            este notebook
├── ROADMAP.MD           especificacao, faixas seguras e fluxogramas
├── analise_assistida_ia.md   classificacao, anomalias e riscos
└── reflexao_critica.md  item 1.6
```